In [8]:
import io
import zipfile

import pandas as pd
import requests

# Download the official UCI dataset
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

response = requests.get(url)
response.raise_for_status()

# Open the downloaded ZIP file
outer_zip = zipfile.ZipFile(io.BytesIO(response.content))

print("Files in downloaded archive:")
print(outer_zip.namelist())

# Find and open the nested bank.zip file
bank_zip_name = next(name for name in outer_zip.namelist() if name.endswith("bank.zip"))

inner_zip = zipfile.ZipFile(io.BytesIO(outer_zip.read(bank_zip_name)))

# Find and load bank-full.csv
csv_name = next(name for name in inner_zip.namelist() if name.endswith("bank-full.csv"))

df = pd.read_csv(io.BytesIO(inner_zip.read(csv_name)), sep=";")

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

[ColabLinter] ✔ Auto-format completed (0.014s)
Files in downloaded archive:
['bank.zip', 'bank-additional.zip']

Dataset loaded successfully!
Dataset shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [9]:

print("Target variable distribution:")
print(df["y"].value_counts())

print("\nTarget percentages:")
print(df["y"].value_counts(normalize=True) * 100)

print("\nDataset information:")
df.info()


Target variable distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target percentages:
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  ob

In [10]:
# 1. Check for duplicate records
print("Duplicate records:", df.duplicated().sum())

# 2. Examine numerical variables
print("\nNumerical variable summary:")
display(df.describe().T)

# 3. Check categorical variables
categorical_columns = df.select_dtypes(include="object").columns

print("\nUnique values in categorical columns:")

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].value_counts().head(10))

# 4. Check special values in the dataset
print("\nUnknown values per column:")

for col in df.columns:
    unknown_count = (df[col] == "unknown").sum()
    if unknown_count > 0:
        print(col, ":", unknown_count)

# 5. Examine the previous campaign contact variable
print("\nPrevious campaign contact (pdays):")
print(df["pdays"].value_counts().head(10))

[ColabLinter] ✔ Auto-format completed (0.018s)
Duplicate records: 0

Numerical variable summary:


,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0



Unique values in categorical columns:

job:
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
Name: count, dtype: int64

marital:
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64

education:
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

default:
default
no     44396
yes      815
Name: count, dtype: int64

housing:
housing
yes    25130
no     20081
Name: count, dtype: int64

loan:
loan
no     37967
yes     7244
Name: count, dtype: int64

contact:
contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

month:
month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579
Name: count, dtype: int64

poutcom

In [11]:
from sklearn.model_selection import train_test_split

# Work on a copy so the original dataset remains unchanged
data = df.copy()

# Create a feature indicating previous campaign contact
data["previously_contacted"] = (data["pdays"] != -1).astype(int)

# Replace -1 with 0, while retaining the new indicator
data["pdays"] = data["pdays"].replace(-1, 0)

# Remove duration to avoid using information only known after the call
data = data.drop(columns=["duration"])

# Convert target into binary values
y = data["y"].map({"no": 0, "yes": 1})

# Separate features from target
X = data.drop(columns=["y"])

# First split: 80% development, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: 75% of development for training, 25% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# Display dataset sizes
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("\nTarget distribution in each set:")

for name, target in [("Training", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"\n{name}:")
    print(target.value_counts(normalize=True).round(4))

[ColabLinter] ✔ Auto-format completed (0.072s)
Training set: (27126, 16)
Validation set: (9042, 16)
Test set: (9043, 16)

Target distribution in each set:

Training:
y
0    0.883
1    0.117
Name: proportion, dtype: float64

Validation:
y
0    0.883
1    0.117
Name: proportion, dtype: float64

Test:
y
0    0.883
1    0.117
Name: proportion, dtype: float64


In [12]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical columns
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

# Build preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# Build the complete model pipeline
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

# Train the model
logistic_model.fit(X_train, y_train)

# Predict on validation data
y_val_pred = logistic_model.predict(X_val)
y_val_prob = logistic_model.predict_proba(X_val)[:, 1]

# Evaluate model performance
print("\nLOGISTIC REGRESSION - VALIDATION RESULTS")

print("Accuracy:", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_pred), 4))
print("Recall:", round(recall_score(y_val, y_val_pred), 4))
print("F1-score:", round(f1_score(y_val, y_val_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_prob), 4))
print("PR-AUC:", round(average_precision_score(y_val, y_val_prob), 4))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

[ColabLinter] ✔ Auto-format completed (0.069s)
Numerical features: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'previously_contacted']

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

LOGISTIC REGRESSION - VALIDATION RESULTS
Accuracy: 0.8945
Precision: 0.6769
Recall: 0.1881
F1-score: 0.2944
ROC-AUC: 0.7691
PR-AUC: 0.4129

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94      7984
           1       0.68      0.19      0.29      1058

    accuracy                           0.89      9042
   macro avg       0.79      0.59      0.62      9042
weighted avg       0.88      0.89      0.87      9042


Confusion Matrix:
[[7889   95]
 [ 859  199]]


In [13]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

# Test different classification thresholds
thresholds = np.arange(0.1, 1.0, 0.05)

results = []

for threshold in thresholds:
    # Convert probabilities into binary predictions
    y_pred_threshold = (y_val_prob >= threshold).astype(int)

    results.append(
        {
            "threshold": round(threshold, 2),
            "precision": precision_score(y_val, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_val, y_pred_threshold, zero_division=0),
            "f1_score": f1_score(y_val, y_pred_threshold, zero_division=0),
        }
    )

# Display results
threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))

# Find threshold with highest F1-score
best_result = threshold_results.loc[threshold_results["f1_score"].idxmax()]

print("\nBEST THRESHOLD:")
print(best_result)

[ColabLinter] ✔ Auto-format completed (0.017s)


,threshold,precision,recall,f1_score
0,0.10,0.2176,0.7146,0.3336
1,0.15,0.3487,0.5435,0.4248
2,0.20,0.4644,0.4433,0.4536
3,0.25,0.5338,0.3809,0.4446
4,0.30,0.5519,0.3318,0.4144
5,0.35,0.5846,0.2873,0.3853
6,0.40,0.6197,0.2495,0.3558
7,0.45,0.6437,0.2117,0.3186
8,0.50,0.6769,0.1881,0.2944
9,0.55,0.7074,0.1531,0.2517



BEST THRESHOLD:
threshold    0.200000
precision    0.464356
recall       0.443289
f1_score     0.453578
Name: 2, dtype: float64


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Preprocessing for Random Forest
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", "passthrough", numerical_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# Build Random Forest pipeline
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Train the model
random_forest_model.fit(X_train, y_train)

# Predict on validation data
y_val_rf_pred = random_forest_model.predict(X_val)
y_val_rf_prob = random_forest_model.predict_proba(X_val)[:, 1]

# Evaluate
print("RANDOM FOREST - VALIDATION RESULTS")

print("Accuracy:", round(accuracy_score(y_val, y_val_rf_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_rf_pred), 4))
print("Recall:", round(recall_score(y_val, y_val_rf_pred), 4))
print("F1-score:", round(f1_score(y_val, y_val_rf_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_rf_prob), 4))
print("PR-AUC:", round(average_precision_score(y_val, y_val_rf_prob), 4))

print("\nClassification Report:")
print(classification_report(y_val, y_val_rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_rf_pred))

[ColabLinter] ✔ Auto-format completed (0.020s)
RANDOM FOREST - VALIDATION RESULTS
Accuracy: 0.8877
Precision: 0.5262
Recall: 0.4083
F1-score: 0.4598
ROC-AUC: 0.7981
PR-AUC: 0.4526

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      7984
           1       0.53      0.41      0.46      1058

    accuracy                           0.89      9042
   macro avg       0.73      0.68      0.70      9042
weighted avg       0.88      0.89      0.88      9042


Confusion Matrix:
[[7595  389]
 [ 626  432]]


In [15]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.10, 0.81, 0.05)

rf_threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_rf_prob >= threshold).astype(int)

    rf_threshold_results.append(
        {
            "threshold": round(threshold, 2),
            "precision": precision_score(y_val, y_pred, zero_division=0),
            "recall": recall_score(y_val, y_pred, zero_division=0),
            "f1_score": f1_score(y_val, y_pred, zero_division=0),
        }
    )

rf_threshold_results = pd.DataFrame(rf_threshold_results)

display(rf_threshold_results.round(4))

# Find the best F1 threshold
best_rf_result = rf_threshold_results.loc[rf_threshold_results["f1_score"].idxmax()]

print("\nBEST RANDOM FOREST THRESHOLD:")
print(best_rf_result)

[ColabLinter] ✔ Auto-format completed (0.040s)


,threshold,precision,recall,f1_score
0,0.10,0.1522,0.9102,0.2607
1,0.15,0.1934,0.8280,0.3135
2,0.20,0.2451,0.7514,0.3696
3,0.25,0.3043,0.6843,0.4213
4,0.30,0.3596,0.6295,0.4577
5,0.35,0.4023,0.5737,0.4729
6,0.40,0.4518,0.5142,0.4810
7,0.45,0.4915,0.4641,0.4774
8,0.50,0.5262,0.4083,0.4598
9,0.55,0.5593,0.3431,0.4253



BEST RANDOM FOREST THRESHOLD:
threshold    0.400000
precision    0.451827
recall       0.514178
f1_score     0.480990
Name: 6, dtype: float64


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

# Apply the selected threshold
rf_final_threshold = 0.40

y_val_rf_tuned = (y_val_rf_prob >= rf_final_threshold).astype(int)

# Confusion matrix
print("CONFUSION MATRIX:")
print(confusion_matrix(y_val, y_val_rf_tuned))

# Classification report
print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_val, y_val_rf_tuned, target_names=["No subscription", "Subscription"]
    )
)

[ColabLinter] ✔ Auto-format completed (0.021s)
CONFUSION MATRIX:
[[7324  660]
 [ 514  544]]

CLASSIFICATION REPORT:
                 precision    recall  f1-score   support

No subscription       0.93      0.92      0.93      7984
   Subscription       0.45      0.51      0.48      1058

       accuracy                           0.87      9042
      macro avg       0.69      0.72      0.70      9042
   weighted avg       0.88      0.87      0.87      9042



In [17]:
from sklearn.model_selection import RandomizedSearchCV

# Define the hyperparameters to test
param_distributions = {
    "classifier__n_estimators": [150, 200, 300],
    "classifier__max_depth": [None, 10, 15, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4, 6],
    "classifier__max_features": ["sqrt", 0.5],
}

# Set up randomized search
random_search = RandomizedSearchCV(
    estimator=random_forest_model,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="average_precision",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

# Train using only the training data
random_search.fit(X_train, y_train)

# Display the best settings
print("BEST HYPERPARAMETERS:")
print(random_search.best_params_)

print("\nBEST CROSS-VALIDATION AVERAGE PRECISION:")
print(random_search.best_score_)

[ColabLinter] ✔ Auto-format completed (0.028s)
Fitting 3 folds for each of 8 candidates, totalling 24 fits
BEST HYPERPARAMETERS:
{'classifier__n_estimators': 200, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 6, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 15}

BEST CROSS-VALIDATION AVERAGE PRECISION:
0.4262925611741031


In [18]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Get the best model
tuned_rf_model = random_search.best_estimator_

# Predict on validation data
y_val_tuned_prob = tuned_rf_model.predict_proba(X_val)[:, 1]

# Evaluate at the default threshold of 0.50
y_val_tuned_pred = (y_val_tuned_prob >= 0.50).astype(int)

print("TUNED RANDOM FOREST - VALIDATION RESULTS")
print("----------------------------------------")
print("Accuracy:", round(accuracy_score(y_val, y_val_tuned_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_tuned_pred), 4))
print("Recall:", round(recall_score(y_val, y_val_tuned_pred), 4))
print("F1-score:", round(f1_score(y_val, y_val_tuned_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_tuned_prob), 4))
print("PR-AUC:", round(average_precision_score(y_val, y_val_tuned_prob), 4))

[ColabLinter] ✔ Auto-format completed (0.017s)
TUNED RANDOM FOREST - VALIDATION RESULTS
----------------------------------------
Accuracy: 0.8477
Precision: 0.3985
Recall: 0.5917
F1-score: 0.4762
ROC-AUC: 0.8039
PR-AUC: 0.4561


In [19]:
thresholds = np.arange(0.10, 0.81, 0.05)

tuned_threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_tuned_prob >= threshold).astype(int)

    tuned_threshold_results.append(
        {
            "threshold": round(threshold, 2),
            "precision": precision_score(y_val, y_pred, zero_division=0),
            "recall": recall_score(y_val, y_pred, zero_division=0),
            "f1_score": f1_score(y_val, y_pred, zero_division=0),
        }
    )

tuned_threshold_results = pd.DataFrame(tuned_threshold_results)

display(tuned_threshold_results.round(4))

best_tuned_result = tuned_threshold_results.loc[
    tuned_threshold_results["f1_score"].idxmax()
]

print("\nBEST TUNED MODEL THRESHOLD:")
print(best_tuned_result)

[ColabLinter] ✔ Auto-format completed (0.024s)


,threshold,precision,recall,f1_score
0,0.10,0.1194,0.9991,0.2133
1,0.15,0.1284,0.9830,0.2271
2,0.20,0.1439,0.9442,0.2498
3,0.25,0.1643,0.8979,0.2777
4,0.30,0.1933,0.8450,0.3146
5,0.35,0.2331,0.7703,0.3579
6,0.40,0.2854,0.6985,0.4053
7,0.45,0.3409,0.6389,0.4446
8,0.50,0.3985,0.5917,0.4762
9,0.55,0.4355,0.5359,0.4805



BEST TUNED MODEL THRESHOLD:
threshold    0.600000
precision    0.477294
recall       0.486767
f1_score     0.481984
Name: 10, dtype: float64


In [20]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Lock the selected threshold
final_threshold = 0.60

# Generate test probabilities
y_test_prob = tuned_rf_model.predict_proba(X_test)[:, 1]

# Convert probabilities into predictions
y_test_pred = (y_test_prob >= final_threshold).astype(int)

# Final test evaluation
print("FINAL TEST RESULTS")
print("===================")

print("Threshold:", final_threshold)
print("Accuracy:", round(accuracy_score(y_test, y_test_pred), 4))
print("Precision:", round(precision_score(y_test, y_test_pred), 4))
print("Recall:", round(recall_score(y_test, y_test_pred), 4))
print("F1-score:", round(f1_score(y_test, y_test_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_test_prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, y_test_prob), 4))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_test_pred))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_test, y_test_pred, target_names=["No subscription", "Subscription"]
    )
)

[ColabLinter] ✔ Auto-format completed (0.020s)
FINAL TEST RESULTS
Threshold: 0.6
Accuracy: 0.8805
Precision: 0.4887
Recall: 0.4688
F1-score: 0.4785
ROC-AUC: 0.8003
PR-AUC: 0.4506

CONFUSION MATRIX
[[7466  519]
 [ 562  496]]

CLASSIFICATION REPORT
                 precision    recall  f1-score   support

No subscription       0.93      0.94      0.93      7985
   Subscription       0.49      0.47      0.48      1058

       accuracy                           0.88      9043
      macro avg       0.71      0.70      0.71      9043
   weighted avg       0.88      0.88      0.88      9043



In [21]:
import joblib

# Package the model and its prediction settings
model_artifact = {
    "model": tuned_rf_model,
    "threshold": final_threshold,
    "features": X_train.columns.tolist(),
    "target_mapping": {"no": 0, "yes": 1},
    "excluded_feature": "duration",
}

# Save the artifact
joblib.dump(model_artifact, "customer_subscription_model.joblib")

print("Model saved successfully!")

# Download the model to your computer
from google.colab import files

files.download("customer_subscription_model.joblib")

[ColabLinter] ✔ Auto-format completed (0.063s)
Model saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
%%writefile app.py

import joblib
import pandas as pd

from fastapi import FastAPI
from pydantic import BaseModel

# Load the saved model
artifact = joblib.load("customer_subscription_model.joblib")

model = artifact["model"]
threshold = artifact["threshold"]
feature_names = artifact["features"]

# Create the FastAPI application
app = FastAPI(
    title="Customer Subscription Prediction API",
    description="Predicts whether a bank customer is likely to subscribe to a term deposit.",
    version="1.0.0"
)


# Define the customer information expected by the API
class Customer(BaseModel):
    age: int
    job: str
    marital: str
    education: str
    default: str
    balance: int
    housing: str
    loan: str
    contact: str
    day: int
    month: str
    campaign: int
    pdays: int
    previous: int
    poutcome: str


@app.get("/")
def home():
    return {
        "message": "Customer Subscription Prediction API is running"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_loaded": True
    }


@app.post("/predict")
def predict(customer: Customer):

    # Convert the customer information into a DataFrame
    customer_data = pd.DataFrame([customer.dict()])

    # Reproduce the same feature engineering used during training
    customer_data["previously_contacted"] = (
        customer_data["pdays"] != -1
    ).astype(int)

    customer_data["pdays"] = customer_data["pdays"].replace(-1, 0)

    # Ensure the columns are in the exact training order
    customer_data = customer_data[feature_names]

    # Predict subscription probability
    probability = model.predict_proba(customer_data)[0, 1]

    # Apply the selected threshold
    prediction = (
        "yes" if probability >= threshold else "no"
    )

    return {
        "subscription_probability": round(float(probability), 4),
        "prediction": prediction,
        "threshold": threshold
    }

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
Writing app.py


In [23]:
%pip install fastapi uvicorn

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off


In [24]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

sample_customer = {
    "age": 35,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 1500,
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "may",
    "campaign": 1,
    "pdays": -1,
    "previous": 0,
    "poutcome": "unknown",
}

response = client.post("/predict", json=sample_customer)

print("Status code:", response.status_code)
print("Prediction response:")
print(response.json())

[ColabLinter] ✔ Auto-format completed (0.022s)
Status code: 200
Prediction response:
{'subscription_probability': 0.4702, 'prediction': 'no', 'threshold': 0.6}


In [25]:
{"subscription_probability": 0.4702, "prediction": "no", "threshold": 0.6}

[ColabLinter] ✔ Auto-format completed (0.034s)


{'subscription_probability': 0.4702, 'prediction': 'no', 'threshold': 0.6}

In [26]:

%%writefile app.py

import joblib
import pandas as pd

from fastapi import FastAPI
from pydantic import BaseModel

# Load the trained model
artifact = joblib.load("customer_subscription_model.joblib")

model = artifact["model"]
threshold = artifact["threshold"]
feature_names = artifact["features"]

# Create API
app = FastAPI(
    title="Customer Subscription Prediction API",
    description="Predicts whether a bank customer is likely to subscribe to a term deposit.",
    version="1.0.0"
)

# Expected customer information
class Customer(BaseModel):
    age: int
    job: str
    marital: str
    education: str
    default: str
    balance: int
    housing: str
    loan: str
    contact: str
    day: int
    month: str
    campaign: int
    pdays: int
    previous: int
    poutcome: str

@app.get("/")
def home():
    return {
        "message": "Customer Subscription Prediction API is running"
    }

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_loaded": True
    }

@app.post("/predict")
def predict(customer: Customer):

    customer_data = pd.DataFrame([customer.dict()])

    # Feature engineering
    customer_data["previously_contacted"] = (
        customer_data["pdays"] != -1
    ).astype(int)

    customer_data["pdays"] = customer_data["pdays"].replace(-1, 0)

    # Match training feature order
    customer_data = customer_data[feature_names]

    # Predict probability
    probability = model.predict_proba(customer_data)[0, 1]

    # Apply threshold
    prediction = (
        "yes" if probability >= threshold else "no"
    )

    return {
        "subscription_probability": round(float(probability), 4),
        "prediction": prediction,
        "threshold": float(threshold)
    }


[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
Overwriting app.py


In [27]:
%pip install fastapi uvicorn httpx

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off


In [28]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

sample_customer = {
    "age": 35,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 1500,
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "may",
    "campaign": 1,
    "pdays": -1,
    "previous": 0,
    "poutcome": "unknown",
}

response = client.post("/predict", json=sample_customer)

print("Status code:", response.status_code)
print("Prediction:", response.json())

[ColabLinter] ✔ Auto-format completed (0.052s)
Status code: 200
Prediction: {'subscription_probability': 0.4702, 'prediction': 'no', 'threshold': 0.6}


In [29]:
health_response = client.get("/health")

print("Health status:", health_response.status_code)
print(health_response.json())

Health status: 200
{'status': 'healthy', 'model_loaded': True}


In [30]:
%%writefile requirements.txt

fastapi
uvicorn
pandas
numpy
scikit-learn
joblib
pydantic

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
Writing requirements.txt


In [31]:
with open("requirements.txt") as file:
    print(file.read())

[ColabLinter] ✔ Auto-format completed (0.022s)

fastapi
uvicorn
pandas
numpy
scikit-learn
joblib
pydantic



In [32]:
pip install -r requirements.txt

[ColabLinter] Formatter failed during auto-format.


In [33]:
%%writefile Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY customer_subscription_model.joblib .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
Writing Dockerfile


In [34]:
import os
import shutil

# Create project directory
project_dir = "customer-subscription-prediction"
os.makedirs(project_dir, exist_ok=True)

# Files required by the application
files_to_copy = [
    "app.py",
    "customer_subscription_model.joblib",
    "requirements.txt",
    "Dockerfile",
]

# Copy files into project directory
for filename in files_to_copy:
    shutil.copy(filename, os.path.join(project_dir, filename))

print("Project files copied successfully!")
print("\nProject directory contents:")

for filename in os.listdir(project_dir):
    print(filename)

[ColabLinter] ✔ Auto-format completed (0.048s)
Project files copied successfully!

Project directory contents:
customer_subscription_model.joblib
requirements.txt
app.py
Dockerfile


In [38]:
%%writefile customer-subscription-prediction/.gitignore

__pycache__/
*.pyc
.ipynb_checkpoints/
.env
venv/
.venv/
.DS_Store

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
Overwriting customer-subscription-prediction/.gitignore


In [39]:
%%writefile customer-subscription-prediction/README.md

# Customer Subscription Prediction Using Machine Learning

## 1. Project Overview

This project develops a machine learning system that predicts whether a bank customer is likely to subscribe to a term deposit.

The project uses the UCI Bank Marketing dataset and compares Logistic Regression with Random Forest classification.

## 2. Problem Statement

Banks conduct marketing campaigns to encourage customers to subscribe to term deposits.

The objective is to use historical customer and campaign information to estimate the likelihood of subscription and support better targeting of marketing efforts.

## 3. Dataset

Source: UCI Machine Learning Repository

Dataset: Bank Marketing

https://archive.ics.uci.edu/dataset/222/bank+marketing

The bank-full dataset contains 45,211 records and 16 input variables, together with a binary target variable.

The target variable is:

- yes: Customer subscribed.
- no: Customer did not subscribe.

The `duration` variable was excluded because call duration is only known after a marketing call and would introduce information leakage into a pre-call prediction system.

## 4. Machine Learning Models

The project evaluates:

1. Logistic Regression
2. Random Forest Classifier

Random Forest was selected as the final model after model comparison and hyperparameter tuning.

## 5. Final Model

Model: Tuned Random Forest

Classification threshold: 0.60

Final test performance:

| Metric | Score |
|---|---:|
| Accuracy | 0.8805 |
| Precision | 0.4887 |
| Recall | 0.4688 |
| F1-score | 0.4785 |
| ROC-AUC | 0.8003 |
| PR-AUC | 0.4506 |

## 6. API

The project uses FastAPI to provide predictions.

Available endpoints:

- GET `/` - API welcome message.
- GET `/health` - API health status.
- POST `/predict` - Predicts customer subscription.

## 7. Running the Project

Install dependencies:

```bash
pip install -r requirements.txt

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
Overwriting customer-subscription-prediction/README.md


In [41]:
!uvicorn app:app --host 0.0.0.0 --port 8000

[ColabLinter] Autofix skipped: System or Magic command detected. To disable autofix, run: %clautofix off
INFO:     Started server process [31354]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [31354]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "uvloop/loop.pyx", line 1509, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1502, in uvloop.loop.Loop.run_until_complete
    self.run_forever()
  File "uvloop/loop.pyx", line 1371, in uvloop.loop.Loop.run_forever
    self._run(mode)
  File "uvloop/loop.pyx", line 551, in uvl

In [42]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)

response = client.get("/health")

print(response.status_code)
print(response.json())

[ColabLinter] ✔ Auto-format completed (0.019s)
200
{'status': 'healthy', 'model_loaded': True}


In [43]:
import os

for root, dirs, files in os.walk("customer-subscription-prediction"):
    print(f"\nFolder: {root}")
    for filename in files:
        print(" -", filename)


Folder: customer-subscription-prediction
 - README.md
 - customer_subscription_model.joblib
 - requirements.txt
 - .gitignore
 - app.py
 - Dockerfile
